# Exercise 1: Rejection ABC

**Goal.** Approximate the posterior $p(\theta \mid x_{obs})$ for a simulator whose likelihood we pretend not to know, using nothing but the ability to run the simulator.

In Exercise 0 we could write down the likelihood and maximize it. Here we can only *sample* from the simulator. Rejection ABC keeps the parameters whose simulations land close to the observation. Exercises 2 and 3 approximate this same posterior with neural density estimators, and Exercise 3 compares all three.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sbi.utils import BoxUniform

## The simulator and the observation

This cell is **identical in Exercises 1, 2 and 3**, so results are comparable across notebooks. Feel free to change the constants, but change them in all three notebooks.

- **Prior:** $\theta \sim \mathcal{U}(\theta_{low}, \theta_{high})$.
- **Simulator:** $x = A\sin(\omega\,\theta) + b\,\theta + \varepsilon$, with $\varepsilon \sim \mathcal{N}\big(0, \sigma(\theta)^2\big)$ and $\sigma(\theta) = 1.5\,(\sin\theta + 1.5)$. The noise level itself depends on $\theta$: the noise is *heteroscedastic*.
- **Observation:** `x_obs` is one simulation at `THETA_TRUE`, generated with its own fixed seed so it is the same in every notebook.
- **Reference posterior:** `grid_posterior` computes the exact posterior numerically on a grid. That is only possible because we wrote the simulator and know its likelihood; in real SBI problems you can't. We use it purely as a reference. Set `SHOW_GROUND_TRUTH = False` to hide it from the plots.

In [ ]:
# ---------------- Shared setup: identical in exercises 1, 2 and 3 ----------------
SEED = 42
torch.manual_seed(SEED)

# Prior: theta ~ Uniform(THETA_LOW, THETA_HIGH), as a PyTorch distribution -- the same object `sbi` needs in Exercise 3
THETA_LOW, THETA_HIGH = -10.5, 10.5
prior = BoxUniform(low=torch.tensor([THETA_LOW]), high=torch.tensor([THETA_HIGH]))

# Simulator: x = A sin(OMEGA theta) + SLOPE theta + noise, with a theta-dependent noise level
A, OMEGA, SLOPE = 7.0, 0.75, 1.0

def mean_x(theta):
    return A * torch.sin(OMEGA * theta) + SLOPE * theta

def noise_std(theta):
    return 1.5 * (torch.sin(theta) + 1.5)

def simulate(theta):
    """theta: a tensor of any shape. Returns x of the same shape, drawn from the current torch RNG state."""
    theta = torch.as_tensor(theta, dtype=torch.float32)
    return mean_x(theta) + noise_std(theta) * torch.randn_like(theta)

# The observation we do inference on
THETA_TRUE = -1.0
torch.manual_seed(SEED)
x_obs = simulate(torch.tensor(THETA_TRUE)).item()

# Numerical ground-truth posterior on a grid (possible only because we know the likelihood)
SHOW_GROUND_TRUTH = True

def grid_posterior(x, n_grid=2001):
    theta_grid = torch.linspace(THETA_LOW, THETA_HIGH, n_grid)
    log_lik = -0.5 * ((x - mean_x(theta_grid)) / noise_std(theta_grid)) ** 2 - torch.log(noise_std(theta_grid))
    density = torch.exp(log_lik - log_lik.max())
    density = density / (density.sum() * (theta_grid[1] - theta_grid[0]))
    return theta_grid.numpy(), density.numpy()

# Plot colors, consistent across notebooks
COLORS = {"abc": "#E69F00", "mdn": "#009E73", "npe": "#0072B2", "truth": "0.3", "prior": "0.75"}

### What the simulator looks like

For any fixed $\theta$, the likelihood $p(x \mid \theta)$ is a single Gaussian: one peak in $x$. The mean of $x$, however, is not a monotonic function of $\theta$. The sine wiggles faster than the line rises, so many different values of $\theta$ produce similar values of $x$. That is why the posterior $p(\theta \mid x)$ will have several peaks.

In [ ]:
theta_line = torch.linspace(THETA_LOW, THETA_HIGH, 1000)
theta_demo = prior.sample((2000,)).squeeze(-1)
x_demo = simulate(theta_demo)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(theta_demo.numpy(), x_demo.numpy(), s=4, c="k", alpha=0.3, label="simulations")
ax.plot(theta_line.numpy(), mean_x(theta_line).numpy(), c="C3", lw=2, label=r"mean of $x$")
ax.fill_between(theta_line.numpy(), (mean_x(theta_line) - 2 * noise_std(theta_line)).numpy(),
                (mean_x(theta_line) + 2 * noise_std(theta_line)).numpy(),
                color="C3", alpha=0.15, label=r"$\pm 2\sigma(\theta)$")
ax.axhline(x_obs, color="k", ls=":", label=r"$x_{obs}$")
ax.axvline(THETA_TRUE, color="k", ls="--", label=r"$\theta_{true}$")
ax.set(xlabel=r"$\theta$", ylabel="x")
ax.legend(loc="upper left", fontsize=8)
plt.show()

crossings = int(torch.sum(torch.diff(torch.sign(mean_x(theta_line) - x_obs)) != 0))
print(f"theta_true = {THETA_TRUE}, x_obs = {x_obs:.3f}")
print(f"The mean curve crosses x_obs {crossings} times within the prior range.")
if crossings < 2:
    print("WARNING: only one branch of the simulator reaches x_obs, so the posterior will have a single peak. "
          "Pick THETA_TRUE near a peak or trough of the mean curve.")

## Rejection ABC

The algorithm:

1. Sample $\theta_i \sim p(\theta)$ from the prior.
2. Simulate $x_i \sim p(x \mid \theta_i)$.
3. Keep $\theta_i$ if the distance $d(x_i, x_{obs}) = |x_i - x_{obs}|$ is smaller than a tolerance $\epsilon$.

The kept samples are draws from $p(\theta \mid |x - x_{obs}| < \epsilon)$, which approaches the true posterior as $\epsilon \to 0$, at the price of rejecting more and more simulations. Because $x$ is 1D, we compare each simulation to the observation directly; no summary statistics are needed.

**Exercise 1a.** Implement `rejection_abc`. Every step can be vectorized: sample all `n_simulations` parameters at once, simulate them all at once, and keep the ones that land close enough.

In [ ]:
def rejection_abc(x_obs: float, epsilon: float, n_simulations: int) -> np.ndarray:
    """Run rejection ABC. Returns the accepted theta samples as a 1D array."""
    # EXERCISE 1a:
    #   1. Draw `n_simulations` parameters from `prior` with prior.sample((n_simulations,)), then .squeeze(-1) to 1D.
    #   2. Simulate one x for each parameter with simulate(theta).
    #   3. Return the thetas (as a numpy array) whose simulation lies within epsilon of x_obs (absolute distance).
    theta = ...
    x_sim = ...
    return ...

In [ ]:
# Sanity checks
samples = rejection_abc(x_obs, epsilon=1.0, n_simulations=10_000)
assert isinstance(samples, np.ndarray) and samples.ndim == 1, "return a 1D numpy array of accepted thetas"
assert 0 < len(samples) < 10_000, "with epsilon = 1, some but not all simulations should be accepted"
assert np.all((samples >= THETA_LOW) & (samples <= THETA_HIGH)), "accepted thetas must lie in the prior range"
assert len(rejection_abc(x_obs, epsilon=1e9, n_simulations=500)) == 500, "a huge epsilon should accept everything"
print("rejection_abc passes the sanity checks.")

In [ ]:
# A first look, using the samples from the sanity check above (epsilon=1.0, n=10,000).
# Rough and noisy, but it already shows the shape of the posterior before we tune epsilon.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(samples, bins=40, range=(THETA_LOW, THETA_HIGH), density=True,
        color=COLORS["abc"], alpha=0.7, label=f"ABC posterior ({len(samples)} samples)")
ax.axhline(1 / (THETA_HIGH - THETA_LOW), color=COLORS["prior"], lw=2, label="prior")
ax.axvline(THETA_TRUE, color="k", ls="--", label=r"$\theta_{true}$")
ax.set(xlabel=r"$\theta$", ylabel="density", title="Sanity-check posterior sample (epsilon=1.0, n=10,000)")
ax.legend(fontsize=8)
plt.show()

### Choosing $\epsilon$

A smaller $\epsilon$ gives a more accurate posterior but accepts fewer simulations.

**Exercise 1b.** Choose `EPSILON` so that fewer than 1% of the simulations are accepted. Shrink $\epsilon$ from the sanity-check value above until the printed acceptance rate drops below 1%.

In [ ]:
N_SIMULATIONS = 100_000

# EXERCISE 1b: set EPSILON so that the acceptance rate comes out under 1%.
# Start from the sanity-check epsilon (1.0) and shrink it, rerunning the cell, until
# the printed acceptance rate is below 1%. Still enough samples for a good histogram.
EPSILON = ...

abc_posterior_samples = rejection_abc(x_obs, EPSILON, N_SIMULATIONS)
acceptance_rate = len(abc_posterior_samples) / N_SIMULATIONS
print(f"epsilon = {EPSILON}: accepted {len(abc_posterior_samples)} of {N_SIMULATIONS} simulations ({acceptance_rate:.2%})")
if not acceptance_rate < 0.01:
    print("The acceptance rate is not under 1%: shrink EPSILON further.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(abc_posterior_samples, bins=60, range=(THETA_LOW, THETA_HIGH), density=True,
        color=COLORS["abc"], alpha=0.7, label="ABC posterior")
ax.axhline(1 / (THETA_HIGH - THETA_LOW), color=COLORS["prior"], lw=2, label="prior")
if SHOW_GROUND_TRUTH:
    ax.plot(*grid_posterior(x_obs), color=COLORS["truth"], lw=1.5, label="true posterior (grid)")
ax.axvline(THETA_TRUE, color="k", ls="--", label=r"$\theta_{true}$")
ax.set(xlabel=r"$\theta$", ylabel="density", title=rf"Rejection ABC, $\epsilon$ = {EPSILON}")
ax.legend(fontsize=8)
plt.show()

# Count the peaks of a lightly smoothed histogram, as a check that the posterior really is multimodal
counts, _ = np.histogram(abc_posterior_samples, bins=60, range=(THETA_LOW, THETA_HIGH))
smooth = np.convolve(counts, np.ones(3) / 3, mode="same")
n_peaks = sum(smooth[i] >= smooth[i - 1] and smooth[i] > smooth[i + 1] and smooth[i] > 0.2 * smooth.max()
              for i in range(1, len(smooth) - 1))
print(f"The ABC posterior has about {n_peaks} major peaks (bumps under 20% of the tallest are ignored).")
if n_peaks < 2:
    print("WARNING: the ABC posterior looks unimodal. Check THETA_TRUE (see the simulator plot above).")

### How $\epsilon$ trades accuracy for simulations

Run ABC at several tolerances with the same simulation budget, and watch how the spread of the posterior and the acceptance rate change together.

In [ ]:
epsilons = [0.1, 0.5, 2.0, 5.0]

fig, axes = plt.subplots(1, len(epsilons), figsize=(15, 3.4), sharex=True)
for ax, eps in zip(axes, epsilons):
    samples = rejection_abc(x_obs, eps, N_SIMULATIONS)
    ax.hist(samples, bins=60, range=(THETA_LOW, THETA_HIGH), density=True, color=COLORS["abc"], alpha=0.7)
    if SHOW_GROUND_TRUTH:
        ax.plot(*grid_posterior(x_obs), color=COLORS["truth"], lw=1.2)
    ax.axvline(THETA_TRUE, color="k", ls="--")
    ax.set(xlabel=r"$\theta$", title=rf"$\epsilon$ = {eps}: {len(samples) / N_SIMULATIONS:.1%} accepted")
plt.tight_layout()
plt.show()

## Questions to think about

1. As $\epsilon$ shrinks, the posterior gets sharper but the acceptance rate drops. Roughly how many simulations would you need for 1,000 accepted samples at $\epsilon = 0.1$? What would that mean for a simulator that takes a minute per run?
2. Why does this posterior have several peaks? Match each peak to a branch of the simulator plot, and explain why the peaks have different widths and heights. (Hint: the noise level depends on $\theta$.)
3. Without rerunning anything: what would happen to the posterior if the noise level were increased everywhere?
4. A colleague brings you a new observation $x_{obs}'$. What do you have to redo? How does that scale if they bring 1,000 observations?